<a href="https://colab.research.google.com/github/vineet-crypto/Stochastic-Interest-Rate-Modelling-and-Prediction/blob/main/Finance_club_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Stochastic Interest Rate Modelling and Prediction
Part A: Data Engineering and Preprocessing
Markdown Cell:
In this section, we import the historical daily bond yields and perform robust data preprocessing. As required, we will handle missing values, normalise outliers, and format the 9 specific maturity tenors (3M to 30Y) for time-series calibration.

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.metrics import r2_score
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

# 1. Load Data
train_df = pd.read_csv('train_data.csv', parse_dates=['Date']).set_index('Date')
test_df = pd.read_csv('test_data.csv', parse_dates=['Date']).set_index('Date')
test_3m = pd.read_csv('test_data_3M.csv', parse_dates=['Date']).set_index('Date')

# 2. Handle missing values (Interpolation + Fill)
train_df = train_df.interpolate(method='time').fillna(method='ffill').fillna(method='bfill')
test_df = test_df.interpolate(method='time').fillna(method='ffill').fillna(method='bfill')
test_3m = test_3m.interpolate(method='time').fillna(method='ffill').fillna(method='bfill')

# 3. Define Maturities mapping (in years)
tenor_map = {
    'ZC025YR': 0.25, 'ZC050YR': 0.50, 'ZC075YR': 0.75,
    'ZC100YR': 1.00, 'ZC200YR': 2.00, 'ZC500YR': 5.00,
    'ZC1000YR': 10.0, 'ZC2000YR': 20.0, 'ZC3000YR': 30.0
}

train_cols = list(train_df.columns)
train_tenors = [tenor_map[c] for c in train_cols]

# Identify overlapping columns available in test data for evaluation
test_cols = [c for c in train_cols if c in test_df.columns]
test_tenors = [tenor_map[c] for c in test_cols]

print("--- Part A: Data Engineering ---")
print(f"Train data shape: {train_df.shape}")
print(f"Test data target shape: {test_df[test_cols].shape}")
print("Data cleaned, missing values interpolated and filled.\n")

FileNotFoundError: [Errno 2] No such file or directory: 'train_data.csv'

Part B: Base CIR Model Implementation & CalibrationMarkdown Cell:
The Cox-Ingersoll-Ross (CIR) model describes the evolution of the instantaneous short rate via a mean-reverting square-root diffusion process. The stochastic differential equation is given by:  $$dr_{t}=\kappa(\theta-r_{t})dt+\sigma\sqrt{r_{t}}dW_{t}$$Where $\kappa$ is the speed of mean reversion, $\theta$ is the long-run mean, and $\sigma$ is the volatility. We ensure rates remain positive by checking the Feller condition $2\kappa\theta\ge\sigma^{2}$. We calibrate these parameters using Ordinary Least Squares (OLS) on the discretized process.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from sklearn.metrics import r2_score

class StandaloneBaseCIRKalmanFilter:
    """
    Part 1: Base CIR Model Implementation & Calibration via EKF.
    Treats the underlying short rate as a latent state variable to filter out
    measurement errors and market noise from the raw 3M tracking series.
    """
    def __init__(self, dt=1/252):
        self.dt = dt           # Daily discrete step mapping (252 trading days/year)
        self.kappa = None      # Mean reversion speed
        self.theta = None      # Long-run mean level
        self.sigma = None      # Volatility coefficient of variance process
        self.sigma_r = None    # Measurement noise standard deviation

    def _compute_coefficients(self, tau, kappa, theta, sigma):
        """Computes the exact continuous-time analytical CIR model coefficients."""
        if tau == 0:
            return 0.0, 0.0
        h = np.sqrt(kappa**2 + 2 * sigma**2)
        exp_h = np.exp(h * tau)
        den = 2 * h + (kappa + h) * (exp_h - 1)

        B = (2 * (exp_h - 1)) / np.maximum(den, 1e-9)
        num_A = 2 * h * np.exp((kappa + h) * tau / 2)
        power = (2 * kappa * theta) / (sigma**2 + 1e-9)
        ln_A = power * np.log(np.maximum(num_A / np.maximum(den, 1e-9), 1e-9))
        return B, ln_A

    def _execute_ekf_forward_pass(self, r_3m_series, kappa, theta, sigma, sigma_r, tau_3m=0.25):
        """Runs the linear tracking update steps of the Extended Kalman Filter."""
        N = len(r_3m_series)
        filtered_states = np.zeros(N)

        # Initialize filter states using the first sample
        r_filtered = r_3m_series[0]
        P_filtered = 0.01

        B_3m, ln_A_3m = self._compute_coefficients(tau_3m, kappa, theta, sigma)
        H = B_3m / tau_3m  # Jacobian matrix element
        R = max(sigma_r**2, 1e-8)

        for t in range(N):
            # --- Time Prediction Step ---
            if t > 0:
                r_pred = r_filtered + kappa * (theta - r_filtered) * self.dt
                r_pred = max(r_pred, 1e-6)  # Enforce non-negativity boundary constraint
                P_pred = ((1.0 - kappa * self.dt) ** 2) * P_filtered + (sigma ** 2) * r_filtered * self.dt
            else:
                r_pred = r_filtered
                P_pred = P_filtered

            # --- Measurement Correction Step ---
            y_3m_pred = (B_3m * r_pred - ln_A_3m) / tau_3m
            innovation = r_3m_series[t] - y_3m_pred
            S = H * P_pred * H + R
            K = (P_pred * H) / S

            r_filtered = r_pred + K * innovation
            filtered_states[t] = max(r_filtered, 1e-6)
            P_filtered = (1.0 - K * H) * P_pred

        return filtered_states

    def fit_base_model(self, r_3m_train):
        """Optimizes parameter dimensions via global Differential Evolution."""
        bounds = [(0.01, 2.5), (0.01, 0.12), (0.01, 0.30), (0.001, 0.05)]

        def loss_function(params):
            kappa, theta, sigma, sigma_r = params
            # Mathematical guard validating the structural Feller Condition
            feller_penalty = 1e6 * max(0, sigma**2 - 2 * kappa * theta)

            states = self._execute_ekf_forward_pass(r_3m_train, kappa, theta, sigma, sigma_r)
            B, ln_A = self._compute_coefficients(0.25, kappa, theta, sigma)
            preds = (B * states - ln_A) / 0.25
            return np.sum((r_3m_train - preds)**2) + feller_penalty

        res = differential_evolution(loss_function, bounds, strategy='best1bin', maxiter=30, seed=42)
        self.kappa, self.theta, self.sigma, self.sigma_r = res.x

        print(f"Base Parameters Calibrated -> κ: {self.kappa:.4f} | θ: {self.theta:.4f} | σ: {self.sigma:.4f}")

    def generate_predictions(self, r_3m_test_series, maturities):
        """Projects out-of-sample term tracking states over the test data vector."""
        clean_r = self._execute_ekf_forward_pass(r_3m_test_series, self.kappa, self.theta, self.sigma, self.sigma_r)

        predictions_matrix = []
        for tau in maturities:
            B, ln_A = self._compute_coefficients(tau, self.kappa, self.theta, self.sigma)
            yield_pred = (B * clean_r - ln_A) / tau if tau != 0 else clean_r
            predictions_matrix.append(yield_pred)
        return np.array(predictions_matrix).T

# =====================================================================
# EXECUTION & INTEGRATED REPORTING PIPELINE FOR PART 1
# =====================================================================
maturities_years = [0.25, 0.5, 0.75, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0]

print("="*60)
print("     STAGE 1: EVALUATING BASE CIR STATE-SPACE MODEL")
print("="*60)

# Isolate training inputs
r_3m_train_data = clean_train.iloc[:, 0].values

# Calibrate independent engine instances
base_engine = StandaloneBaseCIRKalmanFilter(dt=1/252)
base_engine.fit_base_model(r_3m_train_data)  # <--- FIXED LINE

# Run forecasting sequences across out-of-sample data splits
r_3m_test_data = clean_test.iloc[:, 0].values
base_predicted_curves = base_engine.generate_predictions(r_3m_test_data, maturities_years)

# Segment evaluation tenors (6M through 30Y, skipping the 3M input)
base_preds_eval = base_predicted_curves[:, 1:]
actual_test_eval = clean_test.iloc[:, 1:].values

flat_base_pred = base_preds_eval.flatten()
flat_base_actual = actual_test_eval.flatten()

# Filter out potential missing records safely
valid_mask_base = ~np.isnan(flat_base_actual) & ~np.isnan(flat_base_pred)
base_r2_score = r2_score(flat_base_actual[valid_mask_base], flat_base_pred[valid_mask_base])

print("\n" + "-"*50)
print("             BASE CIR MODEL METRIC REPORT")
print("-"*50)
print(f"Base CIR Out-of-Sample R2 Score : {base_r2_score:.5f}")
print("Interpretation: Single-factor models without shifts struggle with long maturities")
print("due to structural changes in term premiums over time.")
print("="*60 + "\n")

     STAGE 1: EVALUATING BASE CIR STATE-SPACE MODEL
Base Parameters Calibrated -> κ: 0.4128 | θ: 0.0187 | σ: 0.1239

--------------------------------------------------
             BASE CIR MODEL METRIC REPORT
--------------------------------------------------
Base CIR Out-of-Sample R2 Score : -0.21567
Interpretation: Single-factor models without shifts struggle with long maturities
due to structural changes in term premiums over time.



Part C: The Prediction Challenge: Yield Curve ConstructionMarkdown Cell:
Using the calibrated parameters $(\kappa,\theta,\sigma)$, we can reconstruct the full yield curve from the 3M rate proxy. The continuously compounded yield for maturity $\tau$ is computed using the closed-form deterministic functions $A(t,T)$ and $B(t,T)$:  $$y(t,\tau)=\frac{B(t,T)r_{t}-ln A(t,T)}{\tau}$$

In [ ]:
from sklearn.metrics import r2_score

def calculate_cir_yield(r_t, tau, kappa, theta, sigma):
    """Computes continuously compounded yields via exact analytical solutions with overflow walls."""
    kappa, sigma, r_t = max(kappa, 1e-4), max(sigma, 1e-4), max(r_t, 1e-5)
    h = np.sqrt(kappa**2 + 2 * sigma**2)

    # Clip exponential inputs safely within float64 boundaries (~700)
    exp_h_tau = np.exp(np.clip(h * tau, None, 700))
    denominator = 2 * h + (kappa + h) * (exp_h_tau - 1)

    if denominator == 0 or np.isinf(denominator):
        return r_t

    B = (2 * (exp_h_tau - 1)) / denominator
    exp_kappa_h = np.exp(np.clip((kappa + h) * tau / 2, None, 700))
    base_A = np.maximum((2 * h * exp_kappa_h) / denominator, 1e-10)

    A = base_A ** np.clip((2 * kappa * theta) / (sigma**2), None, 10000)
    return (B * r_t - np.log(A)) / tau

# Define target tenors mapping years for evaluation (6M through 30Y)
prediction_tenors = maturities_years[1:]
target_columns_test = ['6M', '9M', '1Y', '2Y', '5Y', '10Y', '20Y', '30Y']

base_predictions = []
actual_test_labels = clean_test[target_columns_test].values

# Loop through the testing data using our explicit labeled column names
for _, row in clean_test.iterrows():
    r_t_test = row['3M'] # Ingest ONLY the 3M input rate proxy

    # Reconstruct remaining curves
    day_preds = [calculate_cir_yield(r_t_test, tau, base_cir.kappa, base_cir.theta, base_cir.sigma) for tau in prediction_tenors]
    base_predictions.append(day_preds)

base_predictions = np.array(base_predictions)

# Score baseline accuracy
valid_mask = ~np.isnan(base_predictions).any(axis=1)
base_r2 = r2_score(actual_test_labels[valid_mask], base_predictions[valid_mask])

print("=== Base CIR Prediction Challenge Results ===")
print(f"Target Accuracy Required    : > 0.8500")
print(f"Base Model Out-of-Sample R2 : {base_r2:.4f}")

=== Base CIR Prediction Challenge Results ===
Target Accuracy Required    : > 0.8500
Base Model Out-of-Sample R2 : 0.2832


Part D: Model Improvement & Extensions:
The base CIR model constrains the yield curve to shapes produced by a single factor. To improve upon this, I have outlined the Two-Factor CIR Model. By introducing a second stochastic factor, we can capture different variations in the yield curve, such as level versus slope, drawing inspiration from the Longstaff-Schwartz framework.

---



In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from sklearn.metrics import r2_score

class StandaloneCIRPlusPlusEngine:
    """
    Part 2: Extended CIR++ Model with Time-Dependent Parameters.
    Introduces a deterministic displacement framework alpha(tau) to absorb
    systemic term premium variations across the yield curve.
    """
    def __init__(self, dt=1/252):
        self.dt = dt
        self.kappa = None
        self.theta = None
        self.sigma = None
        self.sigma_r = None
        self.alpha_shifts = []  # Explicit drift adjustment array per asset tenor

    def _compute_coefficients(self, tau, kappa, theta, sigma):
        """Computes continuous-time analytical CIR model factors."""
        if tau == 0:
            return 0.0, 0.0
        h = np.sqrt(kappa**2 + 2 * sigma**2)
        exp_h = np.exp(h * tau)
        den = 2 * h + (kappa + h) * (exp_h - 1)

        B = (2 * (exp_h - 1)) / np.maximum(den, 1e-9)
        num_A = 2 * h * np.exp((kappa + h) * tau / 2)
        power = (2 * kappa * theta) / (sigma**2 + 1e-9)
        ln_A = power * np.log(np.maximum(num_A / np.maximum(den, 1e-9), 1e-9))
        return B, ln_A

    def _execute_ekf_forward_pass(self, r_3m_series, kappa, theta, sigma, sigma_r, tau_3m=0.25):
        """Applies EKF tracking sequences to clean hidden baseline states."""
        N = len(r_3m_series)
        filtered_states = np.zeros(N)
        r_filtered = r_3m_series[0]
        P_filtered = 0.01

        B_3m, ln_A_3m = self._compute_coefficients(tau_3m, kappa, theta, sigma)
        H = B_3m / tau_3m
        R = max(sigma_r**2, 1e-8)

        for t in range(N):
            if t > 0:
                r_pred = r_filtered + kappa * (theta - r_filtered) * self.dt
                r_pred = max(r_pred, 1e-6)
                P_pred = ((1.0 - kappa * self.dt) ** 2) * P_filtered + (sigma ** 2) * r_filtered * self.dt
            else:
                r_pred = r_filtered
                P_pred = P_filtered

            y_3m_pred = (B_3m * r_pred - ln_A_3m) / tau_3m
            innovation = r_3m_series[t] - y_3m_pred
            S = H * P_pred * H + R
            K = (P_pred * H) / S

            r_filtered = r_pred + K * innovation
            filtered_states[t] = max(r_filtered, 1e-6)
            P_filtered = (1.0 - K * H) * P_pred

        return filtered_states

    def fit_extended_system(self, clean_train_df, maturities_array):
        """Calibrates structural limits and initial term structures."""
        actual_yields = clean_train_df.values
        r_3m_train = actual_yields[:, 0]

        # Optimize global parameter mappings using Differential Evolution
        bounds = [(0.01, 2.5), (0.01, 0.12), (0.01, 0.30), (0.001, 0.05)]

        def loss_function(params):
            kappa, theta, sigma, sigma_r = params
            feller_penalty = 1e6 * max(0, sigma**2 - 2 * kappa * theta)
            states = self._execute_ekf_forward_pass(r_3m_train, kappa, theta, sigma, sigma_r)
            B, ln_A = self._compute_coefficients(0.25, kappa, theta, sigma)
            preds = (B * states - ln_A) / 0.25
            return np.sum((r_3m_train - preds)**2) + feller_penalty

        res = differential_evolution(loss_function, bounds, strategy='best1bin', maxiter=30, seed=42)
        self.kappa, self.theta, self.sigma, self.sigma_r = res.x

        # Map the training data states to establish non-linear shift arrays
        clean_train_states = self._execute_ekf_forward_pass(r_3m_train, self.kappa, self.theta, self.sigma, self.sigma_r)

        self.alpha_shifts = []
        for idx, tau in enumerate(maturities_array):
            B, ln_A = self._compute_coefficients(tau, self.kappa, self.theta, self.sigma)
            base_predictions = (B * clean_train_states - ln_A) / tau if tau != 0 else clean_train_states

            # Map structural deviations directly into deterministic parameters
            shift = np.mean(actual_yields[:, idx] - base_predictions)
            self.alpha_shifts.append(shift)

        print(f"CIR++ Parameters Calibrated -> κ: {self.kappa:.4f} | θ: {self.theta:.4f} | σ: {self.sigma:.4f}")
        print("Maturity displacement shifts successfully locked.")

    def generate_extended_predictions(self, r_3m_test_series, maturities):
        """Projects out-of-sample configurations applying Brigo-Mercurio shifts."""
        clean_r = self._execute_ekf_forward_pass(r_3m_test_series, self.kappa, self.theta, self.sigma, self.sigma_r)

        predictions_matrix = []
        for idx, tau in enumerate(maturities):
            B, ln_A = self._compute_coefficients(tau, self.kappa, self.theta, self.sigma)
            base_yield = (B * clean_r - ln_A) / tau if tau != 0 else clean_r

            # Combine the base structural model with the time-dependent parameter adjustments
            extended_yield = base_yield + self.alpha_shifts[idx]
            predictions_matrix.append(extended_yield)

        return np.array(predictions_matrix).T

# =====================================================================
# EXECUTION & INTEGRATED REPORTING PIPELINE FOR PART 2
# =====================================================================
print("="*60)
print("     STAGE 2: EVALUATING EXTENDED CIR++ TERMINAL ENGINE")
print("="*60)

# Instantiate the independent extended engine
extended_engine = StandaloneCIRPlusPlusEngine(dt=1/252)
extended_engine.fit_extended_system(clean_train, maturities_years)

# Extract predictions across testing horizons
r_3m_test_input = clean_test.iloc[:, 0].values
extended_predicted_curves = extended_engine.generate_extended_predictions(r_3m_test_input, maturities_years)

# Segment out the target evaluation layers (6M through 30Y)
ext_preds_eval = extended_predicted_curves[:, 1:]
actual_test_eval = clean_test.iloc[:, 1:].values

flat_ext_pred = ext_preds_eval.flatten()
flat_ext_actual = actual_test_eval.flatten()

# Ensure robust checking configurations against missing records
valid_mask_ext = ~np.isnan(flat_ext_actual) & ~np.isnan(flat_ext_pred)
extended_r2_score = r2_score(flat_ext_actual[valid_mask_ext], flat_ext_pred[valid_mask_ext])

print("\n" + "="*60)
print("          FINANCE CLUB VERIFICATION REPORT")
print("="*60)
print(f"Target Accuracy Benchmark Required   : > 0.8500")
print(f"CIR++ Engine Out-of-Sample R2 Score  : {extended_r2_score:.5f}")

if extended_r2_score >= 0.85:
    print("Verification Status                  : PASSED! CRITERIA SATISFIED ✅")
else:
    print("Verification Status                  : INSUFFICIENT METRIC RE-CHECK CONVERGENCE ❌")
print("="*60)

     STAGE 2: EVALUATING EXTENDED CIR++ TERMINAL ENGINE
CIR++ Parameters Calibrated -> κ: 0.4128 | θ: 0.0187 | σ: 0.1239
Maturity displacement shifts successfully locked.

          FINANCE CLUB VERIFICATION REPORT
Target Accuracy Benchmark Required   : > 0.8500
CIR++ Engine Out-of-Sample R2 Score  : 0.43381
Verification Status                  : INSUFFICIENT METRIC RE-CHECK CONVERGENCE ❌


Part E: Critical Analysis:1. Model Mechanics and Calibration:Sensitivity: The choice of calibration (OLS vs. MLE) significantly impacts $\kappa$ and $\sigma$. OLS is faster but can be biased by discretization errors compared to exact MLE.  Feller Condition: This breaks down during extreme low-rate environments (e.g., post-2008 or COVID-19) where $\sigma$ spikes relative to the product of $\kappa$ and $\theta$. It can be handled by using truncated distributions or shifted CIR models (CIR++).  Mean-Reversion ($\kappa$): A high $\kappa$ implies shocks (like central bank rate hikes) are transient and rates quickly return to the long-run mean, $\theta$.  2. Prediction and Out-of-Sample Performance:Accuracy: The 3M rate proxy reconstructs short-end maturities well, but longer-term maturities (20Y, 30Y) are harder to fit because they rely heavily on term premia and macroeconomic expectations not captured by the instantaneous short rate.  Over/Underestimation: The single-factor CIR systematically struggles to produce inverted yield curves, often overestimating the long end during tight monetary policy periods.  3. Extensions and Modelling Choices:Justification: The Two-Factor model mathematically decouples the short-term fluctuations (slope) from long-term expectations (level).  Estimation Challenges: Introducing a second factor means the state variables (the two independent processes) are not directly observable from a single proxy rate, requiring complex filtering techniques like the Kalman Filter.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from sklearn.metrics import r2_score

# =====================================================================
# PART 1: BASE COX-INGERSOLL-ROSS (CIR) MODEL VIA KALMAN FILTER
# =====================================================================

class BaseCIRKalmanFilter:
    """
    Milestone B: Base CIR Model Implementation & Calibration.
    Uses an Extended Kalman Filter (EKF) to treat the true short rate as a
    latent state variable, filtering micro-structural noise from the 3M yield.
    """
    def __init__(self, dt=1/252):
        self.dt = dt           # Daily time step mapping (252 trading days/year)
        self.kappa = None      # Speed of mean reversion
        self.theta = None      # Long-run mean level
        self.sigma = None      # Volatility coefficient
        self.sigma_r = None    # Measurement noise variance scale

    def _compute_A_B(self, tau, kappa, theta, sigma):
        """Computes the continuous-time analytical CIR B(tau) and ln(A(tau)) coefficients."""
        if tau == 0:
            return 0.0, 0.0
        h = np.sqrt(kappa**2 + 2 * sigma**2)
        exp_h = np.exp(h * tau)
        den = 2 * h + (kappa + h) * (exp_h - 1)

        # Analytical factor loading mapping terms
        B = (2 * (exp_h - 1)) / np.maximum(den, 1e-9)
        num_A = 2 * h * np.exp((kappa + h) * tau / 2)
        power = (2 * kappa * theta) / (sigma**2 + 1e-9)
        ln_A = power * np.log(np.maximum(num_A / np.maximum(den, 1e-9), 1e-9))
        return B, ln_A

    def _run_ekf(self, r_3m_series, kappa, theta, sigma, sigma_r, tau_3m=0.25):
        """Executes the Extended Kalman Filter forward loop to extract hidden states."""
        N = len(r_3m_series)
        filtered_states = np.zeros(N)

        # Initialize state matrix with the first tracking element
        r_filtered = r_3m_series[0]
        P_filtered = 0.01

        B_3m, ln_A_3m = self._compute_A_B(tau_3m, kappa, theta, sigma)
        H = B_3m / tau_3m  # Measurement Jacobian operator
        R = max(sigma_r**2, 1e-8)

        for t in range(N):
            # --- Time Update Step (Prediction) ---
            if t > 0:
                r_pred = r_filtered + kappa * (theta - r_filtered) * self.dt
                r_pred = max(r_pred, 1e-6)  # Boundary guard enforcing positive interest rates
                P_pred = ((1.0 - kappa * self.dt) ** 2) * P_filtered + (sigma ** 2) * r_filtered * self.dt
            else:
                r_pred = r_filtered
                P_pred = P_filtered

            # --- Measurement Update Step (Correction) ---
            y_3m_pred = (B_3m * r_pred - ln_A_3m) / tau_3m
            innovation = r_3m_series[t] - y_3m_pred
            S = H * P_pred * H + R
            K = (P_pred * H) / S

            r_filtered = r_pred + K * innovation
            filtered_states[t] = max(r_filtered, 1e-6)
            P_filtered = (1.0 - K * H) * P_pred

        return filtered_states

    def fit(self, r_3m_train):
        """Calibrates baseline dynamics using global Differential Evolution."""
        # Parameter search boundaries ensuring numerical stability
        bounds = [(0.01, 2.5), (0.01, 0.12), (0.01, 0.30), (0.001, 0.05)]

        def loss_func(params):
            kappa, theta, sigma, sigma_r = params
            # Analytical penalty checking the structural Feller Condition
            penalty = 1e6 * max(0, sigma**2 - 2 * kappa * theta)

            states = self._run_ekf(r_3m_train, kappa, theta, sigma, sigma_r)
            B, ln_A = self._compute_A_B(0.25, kappa, theta, sigma)
            preds = (B * states - ln_A) / 0.25
            return np.sum((r_3m_train - preds)**2) + penalty

        res = differential_evolution(loss_func, bounds, strategy='best1bin', maxiter=30, seed=42)
        self.kappa, self.theta, self.sigma, self.sigma_r = res.x
        print(f"Base Parameters Locked | κ: {self.kappa:.4f} | θ: {self.theta:.4f} | σ: {self.sigma:.4f}")

    def predict_base_curve(self, r_3m_test_series, maturities):
        """Reconstructs out-of-sample tracking vectors using only base structural laws."""
        clean_r = self._run_ekf(r_3m_test_series, self.kappa, self.theta, self.sigma, self.sigma_r)

        preds = []
        for tau in maturities:
            B, ln_A = self._compute_A_B(tau, self.kappa, self.theta, self.sigma)
            yield_pred = (B * clean_r - ln_A) / tau if tau != 0 else clean_r
            preds.append(yield_pred)
        return np.array(preds).T


# =====================================================================
# PART 2: MODEL IMPROVEMENT (TIME-DEPENDENT CIR++ EXTENSION)
# =====================================================================

class CIRPlusPlusExtension(BaseCIRKalmanFilter):
    """
    Milestone D: Model Improvement & Extensions.
    Inherits EKF properties from the base class and applies Brigo-Mercurio
    deterministic displacement shifts to absorb systemic term premium variations.
    """
    def __init__(self, dt=1/252):
        super().__init__(dt)
        self.alpha_shifts = []  # Explicit cross-sectional shift correction factors

    def fit_extension(self, clean_train_df, maturities_array):
        """Calibrates initial term structures by identifying exact shift parameters."""
        print("Calibrating CIR++ Deterministic Shift Term Mapping...")
        actual_yields = clean_train_df.values
        r_3m_train = actual_yields[:, 0]

        # 1. Calibrate base dynamics if not yet established
        if self.kappa is None:
            self.fit(r_3m_train)

        # 2. Extract latent state tracking profiles from the training dataset
        clean_train_states = self._run_ekf(r_3m_train, self.kappa, self.theta, self.sigma, self.sigma_r)

        # 3. Derive shift metrics mapping to average historical yield deviations per tenor
        self.alpha_shifts = []
        for idx, tau in enumerate(maturities_array):
            B, ln_A = self._compute_A_B(tau, self.kappa, self.theta, self.sigma)
            base_predictions = (B * clean_train_states - ln_A) / tau if tau != 0 else clean_train_states

            # Shift function captures systematic drift missing from single-factor baselines
            shift = np.mean(actual_yields[:, idx] - base_predictions)
            self.alpha_shifts.append(shift)

        print("CIR++ Extension Calibration Completed. Shift constraints locked.")

    def predict_extended_curve(self, r_3m_test_series, maturities):
        """Reconstructs the yield curve by combining base structures and displacement vectors."""
        clean_r = self._run_ekf(r_3m_test_series, self.kappa, self.theta, self.sigma, self.sigma_r)

        preds = []
        for idx, tau in enumerate(maturities):
            B, ln_A = self._compute_A_B(tau, self.kappa, self.theta, self.sigma)
            base_yield = (B * clean_r - ln_A) / tau if tau != 0 else clean_r

            # Apply the Brigo-Mercurio continuous yield adjustments
            extended_yield = base_yield + self.alpha_shifts[idx]
            preds.append(extended_yield)

        return np.array(preds).T


# =====================================================================
# INTEGRATED RUNTIME CHALLENGE VALIDATION PIPELINE
# =====================================================================
# Enforce standard tracking variables over historical data formats
# (Assumes clean_train and clean_test dataframes are populated in your notebook)

maturities_years = [0.25, 0.5, 0.75, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0]

# --- STAGE 1: EXECUTE BASE CIR MODEL ---
print("="*60)
print("     STAGE 1: EVALUATING BASE CIR STATE-SPACE MODEL")
print("="*60)

r_3m_train = clean_train.iloc[:, 0].values
base_model = BaseCIRKalmanFilter(dt=1/252)
base_model.fit(r_3m_train)

r_3m_test = clean_test.iloc[:, 0].values
base_predicted_curves = base_model.predict_base_curve(r_3m_test, maturities_years)

# Slice evaluation targets (6M through 30Y tenors, ignoring the 3M input column)
base_preds_eval = base_predicted_curves[:, 1:]
actual_test_eval = clean_test.iloc[:, 1:].values

flat_base_pred = base_preds_eval.flatten()
flat_base_actual = actual_test_eval.flatten()

valid_mask_base = ~np.isnan(flat_base_actual) & ~np.isnan(flat_base_pred)
base_r2_score = r2_score(flat_base_actual[valid_mask_base], flat_base_pred[valid_mask_base])

print("\n" + "-"*50)
print("             BASE CIR MODEL METRIC REPORT")
print("-"*50)
print(f"Base CIR Out-of-Sample R2 Score : {base_r2_score:.5f}")
print("Note: Rigid structural parameters struggle with out-of-sample cross-sections.")
print("="*60 + "\n\n")


# --- STAGE 2: EXECUTE MODEL EXTENSION (CIR++) ---
print("="*60)
print("     STAGE 2: EVALUATING EXTENDED CIR++ TERMINAL ENGINE")
print("="*60)

extended_model = CIRPlusPlusExtension(dt=1/252)
extended_model.fit_extension(clean_train, maturities_years)

extended_predicted_curves = extended_model.predict_extended_curve(r_3m_test, maturities_years)
ext_preds_eval = extended_predicted_curves[:, 1:]

flat_ext_pred = ext_preds_eval.flatten()
flat_ext_actual = actual_test_eval.flatten()

valid_mask_ext = ~np.isnan(flat_ext_actual) & ~np.isnan(flat_ext_pred)
extended_r2_score = r2_score(flat_ext_actual[valid_mask_ext], flat_ext_pred[valid_mask_ext])

print("\n" + "="*60)
print("          FINANCE CLUB VERIFICATION REPORT")
print("="*60)
print(f"Target Accuracy Benchmark Required   : > 0.8500")
print(f"CIR++ Engine Out-of-Sample R2 Score  : {extended_r2_score:.5f}")

if extended_r2_score >= 0.85:
    print("Verification Status                  : PASSED! CRITERIA SATISFIED ✅")
else:
    print("Verification Status                  : INSUFFICIENT METRIC RE-CHECK CONVERGENCE ❌")
print("="*60)

     STAGE 1: EVALUATING BASE CIR STATE-SPACE MODEL
Base Parameters Locked | κ: 0.4128 | θ: 0.0187 | σ: 0.1239

--------------------------------------------------
             BASE CIR MODEL METRIC REPORT
--------------------------------------------------
Base CIR Out-of-Sample R2 Score : -0.21567
Note: Rigid structural parameters struggle with out-of-sample cross-sections.


     STAGE 2: EVALUATING EXTENDED CIR++ TERMINAL ENGINE
Calibrating CIR++ Deterministic Shift Term Mapping...
Base Parameters Locked | κ: 0.4128 | θ: 0.0187 | σ: 0.1239
CIR++ Extension Calibration Completed. Shift constraints locked.

          FINANCE CLUB VERIFICATION REPORT
Target Accuracy Benchmark Required   : > 0.8500
CIR++ Engine Out-of-Sample R2 Score  : 0.43381
Verification Status                  : INSUFFICIENT METRIC RE-CHECK CONVERGENCE ❌
